In [4]:
import os

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
import os

raw_path = "/content/drive/MyDrive/MutualFundAnalytics/Data/Raw"

files = os.listdir(raw_path)
print(f"Total files: {len(files)}")
for f in sorted(files):
    print(f)

Total files: 16
Axis_Bluechip.csv
Copy of 01_fund_master.csv
Copy of 02_nav_history.csv
Copy of 03_aum_by_fund_house.csv
Copy of 04_monthly_sip_inflows.csv
Copy of 05_category_inflows.csv
Copy of 06_industry_folio_count.csv
Copy of 07_scheme_performance.csv
Copy of 08_investor_transactions.csv
Copy of 09_portfolio_holdings.csv
Copy of 10_benchmark_indices.csv
HDFC_TOP100_NAV.csv
ICICI_Bluechip.csv
Kotak_Bluechip.csv
Nippon_LargeCap.csv
SBI_Bluechip.csv


In [7]:
import pandas as pd
import numpy as np
import os

raw_path = "/content/drive/MyDrive/MutualFundAnalytics/Data/Raw"
processed_path = "/content/drive/MyDrive/MutualFundAnalytics/Data/Processed"

os.makedirs(processed_path, exist_ok=True)
print("Folders ready ✅")

Folders ready ✅


In [8]:
nav = pd.read_csv(f"{raw_path}/Copy of 02_nav_history.csv")

print("Shape:", nav.shape)
print("\nColumns:", nav.columns.tolist())
print("\nFirst 3 rows:")
nav.head(3)

Shape: (46000, 3)

Columns: ['amfi_code', 'date', 'nav']

First 3 rows:


,amfi_code,date,nav
0,119551,2022-01-03,54.3856
1,119551,2022-01-04,54.3474
2,119551,2022-01-05,54.6869


In [9]:
print("Data Types:")
print(nav.dtypes)

print("\nMissing Values:")
print(nav.isnull().sum())

print("\nNAV Stats:")
print(nav['nav'].describe())

Data Types:
amfi_code      int64
date          object
nav          float64
dtype: object

Missing Values:
amfi_code    0
date         0
nav          0
dtype: int64

NAV Stats:
count    46000.000000
mean       269.570265
std        577.187060
min         26.136600
25%         69.170425
50%        122.732150
75%        260.338675
max       4268.549700
Name: nav, dtype: float64


In [10]:
# Step 1: Convert date to datetime
nav['date'] = pd.to_datetime(nav['date'], errors='coerce')

# Step 2: Sort by amfi_code + date
nav = nav.sort_values(['amfi_code', 'date']).reset_index(drop=True)

# Step 3: Forward fill missing NAV (for holidays/weekends)
nav['nav'] = nav.groupby('amfi_code')['nav'].ffill()

# Step 4: Remove duplicates
before = len(nav)
nav = nav.drop_duplicates()
after = len(nav)

# Step 5: Validate NAV > 0
invalid = nav[nav['nav'] <= 0]

print("✅ Date type:", nav['date'].dtype)
print("✅ Sorted: True")
print(f"✅ Duplicates removed: {before - after}")
print(f"✅ Invalid NAV rows (<=0): {len(invalid)}")
print(f"✅ Final shape: {nav.shape}")

✅ Date type: datetime64[ns]
✅ Sorted: True
✅ Duplicates removed: 0
✅ Invalid NAV rows (<=0): 0
✅ Final shape: (46000, 3)


In [11]:
nav.to_csv(f"{processed_path}/clean_nav_history.csv", index=False)
print("✅ clean_nav_history.csv saved!")

✅ clean_nav_history.csv saved!


In [12]:
transactions = pd.read_csv(f"{raw_path}/Copy of 08_investor_transactions.csv")

print("Shape:", transactions.shape)
print("\nColumns:", transactions.columns.tolist())
print("\nFirst 3 rows:")
transactions.head(3)

Shape: (32778, 13)

Columns: ['investor_id', 'transaction_date', 'amfi_code', 'transaction_type', 'amount_inr', 'state', 'city', 'city_tier', 'age_group', 'gender', 'annual_income_lakh', 'payment_mode', 'kyc_status']

First 3 rows:


,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status
0,INV003054,2024-01-01,119092,SIP,1834,Telangana,Hyderabad,T30,56+,Female,77.1,UPI,Verified
1,INV002952,2024-01-01,148567,Redemption,392882,Punjab,Amritsar,B30,18-25,Male,7.1,Cheque,Verified
2,INV003420,2024-01-01,118636,SIP,912,Haryana,Faridabad,B30,36-45,Male,47.2,Mandate,Verified


In [13]:
print("transaction_type unique values:")
print(transactions['transaction_type'].unique())

print("\nkyc_status unique values:")
print(transactions['kyc_status'].unique())

print("\nData Types:")
print(transactions[['transaction_date', 'amount_inr', 'transaction_type', 'kyc_status']].dtypes)

print("\nMissing Values:")
print(transactions.isnull().sum())

print("\namount_inr stats:")
print(transactions['amount_inr'].describe())

transaction_type unique values:
['SIP' 'Redemption' 'Lumpsum']

kyc_status unique values:
['Verified' 'Pending']

Data Types:
transaction_date    object
amount_inr           int64
transaction_type    object
kyc_status          object
dtype: object

Missing Values:
investor_id           0
transaction_date      0
amfi_code             0
transaction_type      0
amount_inr            0
state                 0
city                  0
city_tier             0
age_group             0
gender                0
annual_income_lakh    0
payment_mode          0
kyc_status            0
dtype: int64

amount_inr stats:
count     32778.000000
mean     107437.318628
std      150415.905084
min         400.000000
25%        3153.000000
50%       17782.500000
75%      189324.250000
max      597498.000000
Name: amount_inr, dtype: float64


In [14]:
# Step 1: Convert date to datetime
transactions['transaction_date'] = pd.to_datetime(
    transactions['transaction_date'], errors='coerce'
)

# Step 2: Standardise transaction_type (proper capitalisation)
transactions['transaction_type'] = transactions['transaction_type'].str.strip().str.title()

# Step 3: Validate amount > 0
before = len(transactions)
transactions = transactions[transactions['amount_inr'] > 0]
after = len(transactions)

# Step 4: Validate kyc_status values
valid_kyc = ['Verified', 'Pending']
invalid_kyc = transactions[~transactions['kyc_status'].isin(valid_kyc)]

# Step 5: Remove duplicates
transactions = transactions.drop_duplicates()

print("✅ Date type:", transactions['transaction_date'].dtype)
print("✅ Transaction types:", transactions['transaction_type'].unique())
print("✅ KYC values:", transactions['kyc_status'].unique())
print(f"✅ Invalid amount rows removed: {before - after}")
print(f"✅ Invalid KYC rows: {len(invalid_kyc)}")
print(f"✅ Final shape: {transactions.shape}")

✅ Date type: datetime64[ns]
✅ Transaction types: ['Sip' 'Redemption' 'Lumpsum']
✅ KYC values: ['Verified' 'Pending']
✅ Invalid amount rows removed: 0
✅ Invalid KYC rows: 0
✅ Final shape: (32778, 13)


In [15]:
# Fix SIP back to uppercase
transactions['transaction_type'] = transactions['transaction_type'].replace('Sip', 'SIP')

print("✅ Fixed transaction types:", transactions['transaction_type'].unique())

# Save cleaned file
transactions.to_csv(f"{processed_path}/clean_investor_transactions.csv", index=False)
print("✅ clean_investor_transactions.csv saved!")

✅ Fixed transaction types: ['SIP' 'Redemption' 'Lumpsum']
✅ clean_investor_transactions.csv saved!


In [16]:
performance = pd.read_csv(f"{raw_path}/Copy of 07_scheme_performance.csv")

print("Shape:", performance.shape)
print("\nColumns:", performance.columns.tolist())
print("\nData Types:")
print(performance.dtypes)
print("\nMissing Values:")
print(performance.isnull().sum())
print("\nFirst 3 rows:")
performance.head(3)

Shape: (40, 19)

Columns: ['amfi_code', 'scheme_name', 'fund_house', 'category', 'plan', 'return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct', 'benchmark_3yr_pct', 'alpha', 'beta', 'sharpe_ratio', 'sortino_ratio', 'std_dev_ann_pct', 'max_drawdown_pct', 'aum_crore', 'expense_ratio_pct', 'morningstar_rating', 'risk_grade']

Data Types:
amfi_code               int64
scheme_name            object
fund_house             object
category               object
plan                   object
return_1yr_pct        float64
return_3yr_pct        float64
return_5yr_pct        float64
benchmark_3yr_pct     float64
alpha                 float64
beta                  float64
sharpe_ratio          float64
sortino_ratio         float64
std_dev_ann_pct       float64
max_drawdown_pct      float64
aum_crore               int64
expense_ratio_pct     float64
morningstar_rating      int64
risk_grade             object
dtype: object

Missing Values:
amfi_code             0
scheme_name           0
fund_house     

,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade
0,119551,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,Large Cap,Regular,12.42,12.36,14.45,11.49,0.87,0.89,0.88,1.29,14.0,-21.70,14288,1.54,4,Moderate
1,119552,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,Large Cap,Direct,15.25,11.30,14.23,9.52,1.78,0.87,0.81,1.29,14.0,-24.43,1231,0.66,3,Moderate
2,119598,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,Small Cap,Regular,24.56,23.39,20.67,22.16,1.23,0.89,0.94,1.35,25.0,-13.35,19259,1.43,5,Very High


In [18]:
# Step 1: Validate all return columns are numeric
return_cols = ['return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct', 'benchmark_3yr_pct']
print("Return columns numeric check:")
print(performance[return_cols].dtypes)

# Step 2: Flag anomalies (returns outside -50% to +100% range)
anomalies = performance[
    (performance['return_1yr_pct'] < -50) | (performance['return_1yr_pct'] > 100) |
    (performance['return_3yr_pct'] < -50) | (performance['return_3yr_pct'] > 100) |
    (performance['return_5yr_pct'] < -50) | (performance['return_5yr_pct'] > 100)
]
print(f"\nAnomalous return rows: {len(anomalies)}")

# Step 3: Validate expense_ratio range (0.1% to 2.5%)
invalid_expense = performance[
    (performance['expense_ratio_pct'] < 0.1) |
    (performance['expense_ratio_pct'] > 2.5)
]
print(f"\nInvalid expense_ratio rows: {len(invalid_expense)}")
if len(invalid_expense) > 0:
    print(invalid_expense[['scheme_name', 'expense_ratio_pct']])

# Step 4: Remove duplicates
before = len(performance)
performance = performance.drop_duplicates()
after = len(performance)

print(f"\n✅ Duplicates removed: {before - after}")
print(f"✅ Final shape: {performance.shape}")

# Step 5: Save
performance.to_csv(f"{processed_path}/clean_scheme_performance.csv", index=False)
print("✅ clean_scheme_performance.csv saved!")

Return columns numeric check:
return_1yr_pct       float64
return_3yr_pct       float64
return_5yr_pct       float64
benchmark_3yr_pct    float64
dtype: object

Anomalous return rows: 0

Invalid expense_ratio rows: 0

✅ Duplicates removed: 0
✅ Final shape: (40, 19)
✅ clean_scheme_performance.csv saved!


In [19]:
from sqlalchemy import create_engine
import sqlite3

# Database path
db_path = "/content/drive/MyDrive/MutualFundAnalytics/bluestock_mf.db"

# Create engine
engine = create_engine(f"sqlite:///{db_path}")

print("✅ Database engine created!")
print(f"📁 Location: {db_path}")

✅ Database engine created!
📁 Location: /content/drive/MyDrive/MutualFundAnalytics/bluestock_mf.db


In [20]:
# Load cleaned files
nav = pd.read_csv(f"{processed_path}/clean_nav_history.csv")
transactions = pd.read_csv(f"{processed_path}/clean_investor_transactions.csv")
performance = pd.read_csv(f"{processed_path}/clean_scheme_performance.csv")
fund_master = pd.read_csv(f"{raw_path}/Copy of 01_fund_master.csv")
aum = pd.read_csv(f"{raw_path}/Copy of 03_aum_by_fund_house.csv")

# Load into SQLite database
nav.to_sql("fact_nav", engine, if_exists="replace", index=False)
transactions.to_sql("fact_transactions", engine, if_exists="replace", index=False)
performance.to_sql("fact_performance", engine, if_exists="replace", index=False)
fund_master.to_sql("dim_fund", engine, if_exists="replace", index=False)
aum.to_sql("fact_aum", engine, if_exists="replace", index=False)

print("✅ All tables loaded!")
print("\nVerifying row counts:")
print(f"  fact_nav:          {len(nav):>6} rows")
print(f"  fact_transactions: {len(transactions):>6} rows")
print(f"  fact_performance:  {len(performance):>6} rows")
print(f"  dim_fund:          {len(fund_master):>6} rows")
print(f"  fact_aum:          {len(aum):>6} rows")

✅ All tables loaded!

Verifying row counts:
  fact_nav:           46000 rows
  fact_transactions:  32778 rows
  fact_performance:      40 rows
  dim_fund:              40 rows
  fact_aum:              90 rows


In [21]:
import sqlite3

conn = sqlite3.connect(db_path)

queries = {
    "1. Top 5 Funds by AUM": """
        SELECT scheme_name, aum_crore
        FROM fact_performance
        ORDER BY aum_crore DESC
        LIMIT 5
    """,
    "2. Average NAV per Month": """
        SELECT strftime('%Y-%m', date) AS month,
               ROUND(AVG(nav), 2) AS avg_nav
        FROM fact_nav
        GROUP BY month
        ORDER BY month
        LIMIT 10
    """,
    "3. SIP Transactions by Year": """
        SELECT strftime('%Y', transaction_date) AS year,
               COUNT(*) AS sip_count,
               ROUND(SUM(amount_inr), 2) AS total_amount
        FROM fact_transactions
        WHERE transaction_type = 'SIP'
        GROUP BY year
        ORDER BY year
    """,
    "4. Transactions by State": """
        SELECT state,
               COUNT(*) AS total_transactions,
               ROUND(SUM(amount_inr), 2) AS total_amount
        FROM fact_transactions
        GROUP BY state
        ORDER BY total_transactions DESC
        LIMIT 10
    """,
    "5. Funds with Expense Ratio < 1%": """
        SELECT scheme_name, expense_ratio_pct
        FROM fact_performance
        WHERE expense_ratio_pct < 1.0
        ORDER BY expense_ratio_pct ASC
    """,
    "6. Top 5 Funds by 3 Year Return": """
        SELECT scheme_name, return_3yr_pct
        FROM fact_performance
        ORDER BY return_3yr_pct DESC
        LIMIT 5
    """,
    "7. Transaction Count by Type": """
        SELECT transaction_type,
               COUNT(*) AS count,
               ROUND(SUM(amount_inr), 2) AS total_amount
        FROM fact_transactions
        GROUP BY transaction_type
    """,
    "8. Average NAV by Fund (Last Entry)": """
        SELECT amfi_code,
               ROUND(AVG(nav), 2) AS avg_nav
        FROM fact_nav
        GROUP BY amfi_code
        ORDER BY avg_nav DESC
        LIMIT 5
    """,
    "9. KYC Verified vs Pending": """
        SELECT kyc_status,
               COUNT(*) AS count
        FROM fact_transactions
        GROUP BY kyc_status
    """,
    "10. Transactions by Age Group": """
        SELECT age_group,
               COUNT(*) AS total_transactions,
               ROUND(AVG(amount_inr), 2) AS avg_amount
        FROM fact_transactions
        GROUP BY age_group
        ORDER BY total_transactions DESC
    """
}

for title, query in queries.items():
    print(f"\n{'='*50}")
    print(f"Query {title}")
    print('='*50)
    result = pd.read_sql_query(query, conn)
    print(result.to_string(index=False))

conn.close()
print("\n✅ All 10 queries executed!")


Query 1. Top 5 Funds by AUM
                                          scheme_name  aum_crore
Mirae Asset Emerging Bluechip Fund - Regular - Growth      49046
        Kotak Emerging Equity Fund - Regular - Growth      47469
       Nippon India Small Cap Fund - Regular - Growth      43630
           DSP Top 100 Equity Fund - Regular - Growth      41828
                  UTI Mid Cap Fund - Regular - Growth      41728

Query 2. Average NAV per Month
  month  avg_nav
2022-01   207.06
2022-02   207.72
2022-03   209.69
2022-04   211.83
2022-05   212.73
2022-06   213.86
2022-07   213.96
2022-08   215.68
2022-09   218.49
2022-10   219.53

Query 3. SIP Transactions by Year
year  sip_count  total_amount
2024      13958   153233052.0
2025       5758    64000439.0

Query 4. Transactions by State
         state  total_transactions  total_amount
        Punjab                2965   315780459.0
Madhya Pradesh                2931   308312493.0
    Tamil Nadu                2806   315177237.0
       Gu

In [22]:
schema_sql = """
-- Star Schema for Mutual Fund Analytics
-- BlueStock Capstone Project

CREATE TABLE IF NOT EXISTS dim_fund (
    amfi_code INTEGER PRIMARY KEY,
    scheme_name TEXT,
    fund_house TEXT,
    category TEXT,
    plan TEXT
);

CREATE TABLE IF NOT EXISTS dim_date (
    date_id INTEGER PRIMARY KEY AUTOINCREMENT,
    full_date DATE,
    year INTEGER,
    month INTEGER,
    day INTEGER,
    quarter INTEGER,
    is_weekend INTEGER
);

CREATE TABLE IF NOT EXISTS fact_nav (
    nav_id INTEGER PRIMARY KEY AUTOINCREMENT,
    amfi_code INTEGER,
    date DATE,
    nav REAL,
    FOREIGN KEY (amfi_code) REFERENCES dim_fund(amfi_code)
);

CREATE TABLE IF NOT EXISTS fact_transactions (
    transaction_id INTEGER PRIMARY KEY AUTOINCREMENT,
    investor_id TEXT,
    transaction_date DATE,
    amfi_code INTEGER,
    transaction_type TEXT,
    amount_inr REAL,
    state TEXT,
    city TEXT,
    kyc_status TEXT,
    FOREIGN KEY (amfi_code) REFERENCES dim_fund(amfi_code)
);

CREATE TABLE IF NOT EXISTS fact_performance (
    performance_id INTEGER PRIMARY KEY AUTOINCREMENT,
    amfi_code INTEGER,
    return_1yr_pct REAL,
    return_3yr_pct REAL,
    return_5yr_pct REAL,
    expense_ratio_pct REAL,
    aum_crore INTEGER,
    morningstar_rating INTEGER,
    risk_grade TEXT,
    FOREIGN KEY (amfi_code) REFERENCES dim_fund(amfi_code)
);

CREATE TABLE IF NOT EXISTS fact_aum (
    aum_id INTEGER PRIMARY KEY AUTOINCREMENT,
    fund_house TEXT,
    aum_crore REAL
);
"""

schema_path = "/content/drive/MyDrive/MutualFundAnalytics/schema.sql"
with open(schema_path, "w") as f:
    f.write(schema_sql)

print("✅ schema.sql saved!")

✅ schema.sql saved!


In [23]:
queries_sql = """
-- 10 Analytical SQL Queries
-- BlueStock Mutual Fund Analytics

-- 1. Top 5 Funds by AUM
SELECT scheme_name, aum_crore
FROM fact_performance
ORDER BY aum_crore DESC
LIMIT 5;

-- 2. Average NAV per Month
SELECT strftime('%Y-%m', date) AS month,
       ROUND(AVG(nav), 2) AS avg_nav
FROM fact_nav
GROUP BY month
ORDER BY month;

-- 3. SIP Transactions by Year
SELECT strftime('%Y', transaction_date) AS year,
       COUNT(*) AS sip_count,
       ROUND(SUM(amount_inr), 2) AS total_amount
FROM fact_transactions
WHERE transaction_type = 'SIP'
GROUP BY year
ORDER BY year;

-- 4. Transactions by State
SELECT state,
       COUNT(*) AS total_transactions,
       ROUND(SUM(amount_inr), 2) AS total_amount
FROM fact_transactions
GROUP BY state
ORDER BY total_transactions DESC
LIMIT 10;

-- 5. Funds with Expense Ratio < 1%
SELECT scheme_name, expense_ratio_pct
FROM fact_performance
WHERE expense_ratio_pct < 1.0
ORDER BY expense_ratio_pct ASC;

-- 6. Top 5 Funds by 3 Year Return
SELECT scheme_name, return_3yr_pct
FROM fact_performance
ORDER BY return_3yr_pct DESC
LIMIT 5;

-- 7. Transaction Count by Type
SELECT transaction_type,
       COUNT(*) AS count,
       ROUND(SUM(amount_inr), 2) AS total_amount
FROM fact_transactions
GROUP BY transaction_type;

-- 8. Average NAV by Fund
SELECT amfi_code,
       ROUND(AVG(nav), 2) AS avg_nav
FROM fact_nav
GROUP BY amfi_code
ORDER BY avg_nav DESC
LIMIT 5;

-- 9. KYC Verified vs Pending
SELECT kyc_status,
       COUNT(*) AS count
FROM fact_transactions
GROUP BY kyc_status;

-- 10. Transactions by Age Group
SELECT age_group,
       COUNT(*) AS total_transactions,
       ROUND(AVG(amount_inr), 2) AS avg_amount
FROM fact_transactions
GROUP BY age_group
ORDER BY total_transactions DESC;
"""

queries_path = "/content/drive/MyDrive/MutualFundAnalytics/queries.sql"
with open(queries_path, "w") as f:
    f.write(queries_sql)

print("✅ queries.sql saved!")

✅ queries.sql saved!


In [24]:
data_dictionary = """# Data Dictionary
## BlueStock Mutual Fund Analytics — Day 2

---

## Table: fact_nav
| Column | Type | Description |
|--------|------|-------------|
| amfi_code | INTEGER | Unique fund identifier from AMFI |
| date | DATE | NAV date |
| nav | REAL | Net Asset Value in INR |

---

## Table: fact_transactions
| Column | Type | Description |
|--------|------|-------------|
| investor_id | TEXT | Unique investor identifier |
| transaction_date | DATE | Date of transaction |
| amfi_code | INTEGER | Fund identifier |
| transaction_type | TEXT | SIP / Lumpsum / Redemption |
| amount_inr | REAL | Transaction amount in INR |
| state | TEXT | Investor's state |
| city | TEXT | Investor's city |
| city_tier | TEXT | T30 or B30 city classification |
| age_group | TEXT | Investor age group |
| gender | TEXT | Investor gender |
| annual_income_lakh | REAL | Annual income in lakhs |
| payment_mode | TEXT | UPI / Cheque / Mandate |
| kyc_status | TEXT | Verified or Pending |

---

## Table: fact_performance
| Column | Type | Description |
|--------|------|-------------|
| amfi_code | INTEGER | Fund identifier |
| scheme_name | TEXT | Full name of the scheme |
| fund_house | TEXT | AMC name |
| category | TEXT | Large Cap / Mid Cap / Small Cap etc |
| return_1yr_pct | REAL | 1 year return percentage |
| return_3yr_pct | REAL | 3 year return percentage |
| return_5yr_pct | REAL | 5 year return percentage |
| expense_ratio_pct | REAL | Annual expense ratio (0.1% to 2.5%) |
| aum_crore | INTEGER | Assets Under Management in crores |
| morningstar_rating | INTEGER | Rating from 1 to 5 |
| risk_grade | TEXT | Low / Moderate / High / Very High |

---

## Table: dim_fund
| Column | Type | Description |
|--------|------|-------------|
| amfi_code | INTEGER | Primary key, unique fund ID |
| scheme_name | TEXT | Full scheme name |
| fund_house | TEXT | AMC / fund house name |
| category | TEXT | Fund category |
| plan | TEXT | Regular or Direct plan |

---

## Table: fact_aum
| Column | Type | Description |
|--------|------|-------------|
| fund_house | TEXT | AMC name |
| aum_crore | REAL | Total AUM in crores |

---

## Sources
- AMFI India (amfiindia.com)
- BlueStock Capstone Project datasets
"""

dict_path = "/content/drive/MyDrive/MutualFundAnalytics/data_dictionary.md"
with open(dict_path, "w") as f:
    f.write(data_dictionary)

print("✅ data_dictionary.md saved!")

✅ data_dictionary.md saved!


In [25]:
files_to_check = [
    "/content/drive/MyDrive/MutualFundAnalytics/schema.sql",
    "/content/drive/MyDrive/MutualFundAnalytics/queries.sql",
    "/content/drive/MyDrive/MutualFundAnalytics/data_dictionary.md",
    "/content/drive/MyDrive/MutualFundAnalytics/bluestock_mf.db"
]

for f in files_to_check:
    exists = "✅ Found" if os.path.exists(f) else "❌ Missing"
    print(f"{exists} — {f.split('/')[-1]}")

✅ Found — schema.sql
✅ Found — queries.sql
✅ Found — data_dictionary.md
✅ Found — bluestock_mf.db
